# 1. Genome mCG compartment

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob
from scipy.stats import pearsonr

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.metrics import pairwise_distances, roc_auc_score

from ALLCools.mcds import MCDS
import cooler

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
# L1annot = pd.Series(L1annot).sort_values()

In [ ]:
# group_meta = pd.read_csv(f'{indir}subtype_meta.tsv', sep='\t', index_col=0)
# group_meta = group_meta[['L2_any', 'L1', 'count']]
# group_meta['L1_annot'] = group_meta['L1_annot'].str.replace(' ','-').str.replace('/','_')
# annot2L1 = group_meta[['L1','L1_annot']].set_index('L1_annot')['L1'].to_dict()
# L1annot = group_meta[['L1','L1_annot']].set_index('L1')['L1_annot'].to_dict()


## Get mCG level

In [ ]:
mcds = MCDS.open(f'{indir}merged_allc/subtype1k.mcds', var_dim='chrom1k')
mcds

In [ ]:
bin_df = mcds[['chrom1k_chrom', 'chrom1k_start', 'chrom1k_end']].to_pandas()
bin_df['chrom100k'] = bin_df['chrom1k_chrom'].astype(str) + '_' + (bin_df['chrom1k_start'] // 100000).astype(str)
bin_df

In [ ]:
mcds = mcds.assign_coords(L1=('cell', mcds.get_index('cell').str.split('-').str[0]))
mcds = mcds.groupby('L1').sum()


In [ ]:
# mcds = mcds.assign_coords(chrom10k=('chrom1k', bin_df['chrom1k']))
mcds = mcds.sel({'mc_type':'CGN'})
mcds = MCDS(mcds, obs_dim='L1', var_dim='chrom1k')
mcds

In [ ]:
exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [ ]:
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='L1').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>50]})

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)


In [ ]:
# global_cg = mcds['chrom1k_da'].sum(dim='chrom1k').to_pandas()
# global_cg = global_cg['mc'] / global_cg['cov']

In [ ]:
# global_cg.to_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=False)


In [ ]:
global_cg = pd.read_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=None, index_col=0)


In [ ]:
mc = mcds.sel({'count_type':'mc'})['chrom1k_da'].to_pandas()
cov = mcds.sel({'count_type':'cov'})['chrom1k_da'].to_pandas()

In [ ]:
bin_df = bin_df.loc[mc.columns]
mc = mc.T.groupby(bin_df['chrom100k']).sum()
cov = cov.T.groupby(bin_df['chrom100k']).sum()


In [ ]:
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
mcg = calculate_posterior_mc_frac(mc.T.values, cov.T.values, normalize_per_cell=False)
mcg = pd.DataFrame(mcg, index=mc.columns, columns=mc.index)


In [ ]:
mcg.to_hdf('mCG_distribution/L1_chrom100k_mCG.hdf', key='data')


In [ ]:
mcg = pd.read_hdf('mCG_distribution/L1_chrom100k_mCG.hdf', key='data')
mcg = mcg.T


In [ ]:
mode = 'raw'
comp = pd.read_hdf(f'compartment/L1/comp.{mode}.hdf', key='data')
comp.index = comp['chrom'].astype(str) + '_' + (comp['start'] // 100000).astype(str)
comp = comp.loc[comp['raw_binfilter']]


In [ ]:
selb = comp.index.intersection(mcg.index)
print(len(selb))

In [ ]:
comp = comp.loc[selb]
mcg = mcg.loc[selb]

In [ ]:
## impute
corr = pd.DataFrame([[L1_annot[ct], pearsonr(mcg[ct], comp[ct])[0]] for ct in L1_meta.index], index=L1_meta.index, columns=['L1', 'corr'])
corr = corr.sort_values('corr')
corr    

In [ ]:
## raw
corr = pd.DataFrame([[L1_annot[ct], pearsonr(mcg[ct], comp[ct])[0]] for ct in L1_meta.index], index=L1_meta.index, columns=['L1', 'corr'])
corr = corr.sort_values('corr')
corr    

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4.5), dpi=300)
for i,ct in enumerate(list(corr.index[:4]) + list(corr.index[-4:])):
    ax = axes.flatten()[i]
    ax.scatter(mcg[ct], comp[ct], s=0.1, edgecolor='none', rasterized=True)
    ax.set_title(f'{L1_annot[ct]}\n{np.around(corr.loc[ct,"corr"], decimals=3)}')
    xmin, xmax = np.percentile(mcg[ct], [5, 99])
    shift = (xmax-xmin)*0.1
    ax.set_xlim([xmin-shift, xmax+shift])
    ymin, ymax = np.percentile(comp[ct], [1, 99])
    shift = (ymax-ymin)*0.1
    ax.set_ylim([ymin-shift, ymax+shift])

for ax in axes[:,0]:
    ax.set_ylabel('Comp Score')

fig.tight_layout()
fig.savefig(f'compartment/mCG_comp_{mode}_corr_example_scatter.pdf', transparent=True)



In [ ]:
fig, ax = plt.subplots(figsize=(6, 2), dpi=300)
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.bar(x=np.arange(corr.shape[0]), height=corr['corr'], color=corr.index.map(L1_color), width=0.8)

        
xticks = np.arange(corr.shape[0])
inward_ticks = xticks[corr['corr']<0]
ax.set_xticks(xticks)
ax.tick_params(axis='x', direction='in')
ax.set_xticklabels(corr.index.map(L1_annot), fontsize=8, rotation=90)
xticklabels = ax.get_xticklabels()
for i, tick in enumerate(xticks):
    if tick in inward_ticks:
        xticklabels[i].set_va('bottom')
        xticklabels[i].set_y(0.1)

ax.set_xlim([-0.5, corr.shape[0]-0.5])
ax.set_ylabel('Pearson r')
fig.tight_layout()
fig.savefig(f'compartment/mCG_comp_{mode}_corr.pdf', transparent=True)


## Get mCH level

In [ ]:
mcds = MCDS.open(f'{indir}merged_allc/subtype1k.mcds', var_dim='chrom1k')
mcds

In [ ]:
bin_df = mcds[['chrom1k_chrom', 'chrom1k_start', 'chrom1k_end']].to_pandas()
bin_df['chrom100k'] = bin_df['chrom1k_chrom'].astype(str) + '_' + (bin_df['chrom1k_start'] // 100000).astype(str)
bin_df

In [ ]:
mcds = mcds.assign_coords(L1=('cell', mcds.get_index('cell').str.split('-').str[0]))
mcds = mcds.groupby('L1').sum()


In [ ]:
# mcds = mcds.assign_coords(chrom10k=('chrom1k', bin_df['chrom1k']))
mcds = mcds.sel({'mc_type':'CHN'})
mcds = MCDS(mcds, obs_dim='L1', var_dim='chrom1k')
mcds

In [ ]:
exclude_chromosome = ['chrX', 'chrY', 'chrM', 'chrL']
mcds = mcds.remove_chromosome(exclude_chromosome)


In [ ]:
cov = mcds['chrom1k_da'].sel(count_type='cov').mean(dim='L1').squeeze().to_pandas()


In [ ]:
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(cov, bins=100, log_scale=(True, False), ax=ax)
# ax.set_yscale('log')

In [ ]:
mcds = mcds.sel({'chrom1k':cov.index[cov>1000]})

black_list_path = '/large_experiments/zhoulab/ref/blacklist/hg38-blacklist.v2.bed.gz'
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path, f=0.5
)


In [ ]:
global_cg = mcds['chrom1k_da'].sum(dim='chrom1k').to_pandas()
global_cg = global_cg['mc'] / global_cg['cov']

In [ ]:
global_cg.to_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=False)


In [ ]:
global_cg = pd.read_csv('mCG_distribution/L1_global_mCG.tsv.gz', sep='\t', header=None, index_col=0)


In [ ]:
mc = mcds.sel({'count_type':'mc'})['chrom1k_da'].to_pandas()
cov = mcds.sel({'count_type':'cov'})['chrom1k_da'].to_pandas()

In [ ]:
bin_df = bin_df.loc[mc.columns]
mc = mc.T.groupby(bin_df['chrom100k']).sum()
cov = cov.T.groupby(bin_df['chrom100k']).sum()


In [ ]:
from ALLCools.mcds.utilities import calculate_posterior_mc_frac
mcg = calculate_posterior_mc_frac(mc.T.values, cov.T.values, normalize_per_cell=False)
mcg = pd.DataFrame(mcg, index=mc.columns, columns=mc.index)


In [ ]:
mcg.to_hdf('mCH_distribution/L1_chrom100k_mCH.hdf', key='data')


In [ ]:
mcg = pd.read_hdf('mCH_distribution/L1_chrom100k_mCH.hdf', key='data')
mcg = mcg.T


In [ ]:
mode = 'impute'
comp = pd.read_hdf(f'compartment/L1/comp.{mode}.hdf', key='data')
comp.index = comp['chrom'].astype(str) + '_' + (comp['start'] // 100000).astype(str)
comp = comp.loc[comp['raw_binfilter']]


In [ ]:
selb = comp.index.intersection(mcg.index)
print(len(selb))

In [ ]:
comp = comp.loc[selb]
mcg = mcg.loc[selb]

In [ ]:
## impute
corr = pd.DataFrame([[L1_annot[ct], pearsonr(mcg[ct], comp[ct])[0]] for ct in L1_meta.index], index=L1_meta.index, columns=['L1', 'corr'])
corr = corr.sort_values('corr')
corr

In [ ]:
## raw
corr = pd.DataFrame([[L1_annot[ct], pearsonr(mcg[ct], comp[ct])[0]] for ct in L1_meta.index], index=L1_meta.index, columns=['L1', 'corr'])
corr = corr.sort_values('corr')
corr

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4.5), dpi=300)
for i,ct in enumerate(list(corr.index[:2]) + list(corr.index[-6:])):
    ax = axes.flatten()[i]
    ax.scatter(mcg[ct], comp[ct], s=0.1, edgecolor='none', rasterized=True)
    ax.set_title(f'{L1_annot[ct]}\n{np.around(corr.loc[ct,"corr"], decimals=3)}')
    xmin, xmax = np.percentile(mcg[ct], [5, 99])
    shift = (xmax-xmin)*0.1
    ax.set_xlim([xmin-shift, xmax+shift])
    ymin, ymax = np.percentile(comp[ct], [1, 99])
    shift = (ymax-ymin)*0.1
    ax.set_ylim([ymin-shift, ymax+shift])

for ax in axes[:,0]:
    ax.set_ylabel('Comp Score')

fig.tight_layout()
fig.savefig(f'compartment/mCH_comp_{mode}_corr_example_scatter.pdf', transparent=True)



In [ ]:
fig, ax = plt.subplots(figsize=(6, 2), dpi=300)
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.bar(x=np.arange(corr.shape[0]), height=corr['corr'], color=corr.index.map(L1_color), width=0.8)

        
xticks = np.arange(corr.shape[0])
inward_ticks = xticks[corr['corr']<0]
ax.set_xticks(xticks)
ax.tick_params(axis='x', direction='in')
ax.set_xticklabels(corr.index.map(L1_annot), fontsize=8, rotation=90)
xticklabels = ax.get_xticklabels()
for i, tick in enumerate(xticks):
    if tick in inward_ticks:
        xticklabels[i].set_va('bottom')
        xticklabels[i].set_y(0.1)

ax.set_xlim([-0.5, corr.shape[0]-0.5])
ax.set_ylabel('Pearson r')
fig.tight_layout()
fig.savefig(f'compartment/mCH_comp_{mode}_corr.pdf', transparent=True)


In [ ]:
corr['mCHFrac'] = meta.groupby('L1')['mCHFrac'].mean()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5), dpi=300)
ax.scatter(corr['mCHFrac'], corr['corr'], c=L1meta.loc[corr.index, 'color'], s=10, edgecolor='none')
ax.set_ylabel('Pearson r \nComp. Score vs mCH/CH', fontsize=10)
ax.set_xlabel('Global mCH/CH', fontsize=10)

for i,xx in enumerate(corr.sort_values('corr').index):
    yy = np.arange(-0.35, 0.75, 1.1/35)[i]
    ax.scatter(0.035, yy, c=L1meta.loc[xx, 'color'], s=6, edgecolor='none')
    ax.text(0.038, yy, L1meta.loc[xx, 'L1_annot'], ha='left', va='center', fontsize=4)
    
plt.tight_layout()
